In [ ]:
# Importar librerías estándar para análisis numérico y visualización
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# TensorFlow / Keras: capas y utilidades para construir la GAN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Reshape, Input, BatchNormalization,
    LeakyReLU, Dropout, Conv2DTranspose, Conv2D
)
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import fashion_mnist

In [ ]:
# Fijar semillas para reproducibilidad de resultados
np.random.seed(42)
tf.random.set_seed(42)

### Datos

In [ ]:
# Cargar Fashion MNIST: 60,000 imágenes de entrenamiento, 28×28 en escala de grises
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

In [ ]:
# Visualizar un ejemplo del dataset
plt.imshow(X_train[0], cmap='gray')
plt.axis('off');

In [ ]:
# Verificar el rango original de los píxeles (0-255)
print(f'Valor máximo: {X_train.max()}')
print(f'Valor mínimo: {X_train.min()}')

### Preprocesamiento

In [ ]:
# Normalizar al rango [0, 1] y cast a float32
X_train = X_train.astype('float32') / 255.0

In [ ]:
# Agregar dimensión del canal (escala de grises = 1) para compatibilidad con Conv2D
X_train = X_train.reshape(-1, 28, 28, 1)

In [ ]:
# Escalar al rango [-1, 1] para que coincida con la activación tanh del generador
X_train = X_train * 2 - 1

In [ ]:
# Verificar el nuevo rango de los datos
print(f'Valor máximo: {X_train.max():.2f}')
print(f'Valor mínimo: {X_train.min():.2f}')

In [ ]:
# Hiperparámetros del entrenamiento
buffer_size = 60000
batch_size = 128
latent_dim = 100
epochs = 50

In [ ]:
# Crear dataset de TensorFlow: mezclar y agrupar en batches
train_dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(buffer_size).batch(batch_size)

### Arquitecturas

In [ ]:
# Generador: convierte un vector de ruido latente en una imagen sintética 28×28
# Arquitectura tipo DCGAN: Dense → Reshape → Conv2DTranspose (upsampling)
generator = Sequential([
    Input(shape=(latent_dim,)),
    Dense(units=7*7*256, use_bias=False),
    BatchNormalization(),
    LeakyReLU(0.2),
    Reshape((7, 7, 256)),
    Conv2DTranspose(128, (5, 5), strides=1, padding='same', use_bias=False),
    BatchNormalization(),
    LeakyReLU(0.2),
    Conv2DTranspose(64, (5, 5), strides=2, padding='same', use_bias=False),
    BatchNormalization(),
    LeakyReLU(0.2),
    Conv2DTranspose(1, (5, 5), strides=2, padding='same', use_bias=False, activation='tanh')
])

In [ ]:
# Discriminador: clasifica si una imagen es real (dataset) o falsa (generada)
# Arquitectura tipo DCGAN: Conv2D (downsampling) → LeakyReLU → Dropout → Dense(1)
discriminator = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(64, (5, 5), strides=2, padding='same'),
    LeakyReLU(0.2),
    Dropout(0.3),
    Conv2D(128, (5, 5), strides=2, padding='same'),
    LeakyReLU(0.2),
    Dropout(0.3),
    Flatten(),
    Dense(1)
])

In [ ]:
# Verificar las dimensiones de salida de ambos modelos con datos dummy
test_noise = tf.random.normal([1, latent_dim])
generated_image = generator(test_noise, training=False)
print(f'Salida del generador: {generated_image.shape}')

test_image = tf.random.normal([1, 28, 28, 1])
decision = discriminator(test_image)
print(f'Salida del discriminador: {decision.shape}')

### Pérdidas y Optimizadores

In [ ]:
# BinaryCrossentropy con from_logits=True porque el discriminador no usa activación final
cross_entropy = BinaryCrossentropy(from_logits=True)

In [ ]:
# Pérdida del discriminador: clasificar correctamente reales como 1 y falsas como 0
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

In [ ]:
# Pérdida del generador: engañar al discriminador para que clasifique las falsas como 1
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
# Optimizadores Adam con learning rate conservador para estabilidad en GANs
generator_optimizer = Adam(learning_rate=0.0001)
discriminator_optimizer = Adam(learning_rate=0.0001)

### Entrenamiento

In [ ]:
# Paso de entrenamiento optimizado con tf.function (compilación a grafo)
@tf.function
def train_step(images):
    noise = tf.random.normal([batch_size, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# Función auxiliar para visualizar una cuadrícula de imágenes generadas
def generate_images(model, seed):
    predictions = model(seed, training=False)
    fig = plt.figure(figsize=(8, 8))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        # Deshacer la escala [-1, 1] → [0, 255] para visualización
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Bucle principal de entrenamiento
# Se calcula el promedio de pérdidas por época para un seguimiento más estable
gen_loss_history = []
disc_loss_history = []
seed = tf.random.normal([16, latent_dim])

for epoch in range(epochs):
    epoch_gen_loss = 0
    epoch_disc_loss = 0
    num_batches = 0

    for image_batch in train_dataset:
        gen_loss, disc_loss = train_step(image_batch)
        epoch_gen_loss += gen_loss
        epoch_disc_loss += disc_loss
        num_batches += 1

    avg_gen_loss = epoch_gen_loss / num_batches
    avg_disc_loss = epoch_disc_loss / num_batches
    gen_loss_history.append(avg_gen_loss)
    disc_loss_history.append(avg_disc_loss)

    print(f'Época {epoch + 1:02d} | Gen Loss: {avg_gen_loss:.4f} | Disc Loss: {avg_disc_loss:.4f}')

    # Mostrar imágenes generadas cada 10 épocas
    if (epoch + 1) % 10 == 0:
        print('--- Imágenes generadas ---')
        generate_images(generator, seed)

### Evaluación

In [ ]:
# Graficar la evolución de las pérdidas durante el entrenamiento
plt.figure(figsize=(10, 5))
plt.plot(gen_loss_history, label='Generator Loss')
plt.plot(disc_loss_history, label='Discriminator Loss')
plt.legend()
plt.show()

In [ ]:
# Generar imágenes finales con el modelo entrenado
final_seed = tf.random.normal([16, latent_dim])
generate_images(generator, final_seed)